# Midsem Q4: BPE (Byte Pair Encoding) with 2 Merges

**Corpus:**
```
low lower low lower lowest
low lowest low lowest
low new
newest widest widest
new widest
```

Apply BPE for **2 merges**. Show the vocabulary and the merge rules. How does BPE handle out-of-vocabulary words?

In [ ]:
from collections import Counter, defaultdict
import re

# ─────────────────────────────────────────────
# Step 0: Build word frequency table from corpus
# ─────────────────────────────────────────────
corpus = [
    "low lower low lower lowest",
    "low lowest low lowest",
    "low new",
    "newest widest widest",
    "new widest",
]

word_freq = Counter()
for line in corpus:
    for word in line.split():
        word_freq[word] += 1

print("Word Frequencies in Corpus:")
print("─" * 30)
for word, freq in sorted(word_freq.items()):
    print(f"  '{word}' : {freq}")
print(f"\nTotal unique words: {len(word_freq)}")

In [ ]:
# ─────────────────────────────────────────────
# Step 1: Character-level tokenization with </w> end marker
# ─────────────────────────────────────────────

def word_to_chars(word):
    """Split word into characters with </w> end-of-word marker."""
    return tuple(list(word) + ['</w>'])

# Build initial splits: each word -> tuple of characters + </w>
splits = {}
for word, freq in word_freq.items():
    splits[word_to_chars(word)] = freq

print("Initial Character-Level Representation:")
print("─" * 50)
for token_tuple, freq in sorted(splits.items(), key=lambda x: -x[1]):
    print(f"  {' '.join(token_tuple):30s} (freq: {freq})")

initial_vocab = set()
for token_tuple in splits:
    for tok in token_tuple:
        initial_vocab.add(tok)

print(f"\nInitial Vocabulary ({len(initial_vocab)} tokens):")
print(f"  {sorted(initial_vocab)}")

In [ ]:
# ─────────────────────────────────────────────
# BPE Algorithm: 2 Merges
# ─────────────────────────────────────────────

def count_pairs(splits):
    """Count frequency of all adjacent symbol pairs."""
    pair_counts = defaultdict(int)
    for token_tuple, freq in splits.items():
        for i in range(len(token_tuple) - 1):
            pair = (token_tuple[i], token_tuple[i + 1])
            pair_counts[pair] += freq
    return pair_counts

def merge_pair(splits, best_pair):
    """Merge the most frequent pair in all words."""
    new_splits = {}
    merged_token = best_pair[0] + best_pair[1]
    for token_tuple, freq in splits.items():
        new_tuple = []
        i = 0
        while i < len(token_tuple):
            if i < len(token_tuple) - 1 and token_tuple[i] == best_pair[0] and token_tuple[i + 1] == best_pair[1]:
                new_tuple.append(merged_token)
                i += 2
            else:
                new_tuple.append(token_tuple[i])
                i += 1
        new_splits[tuple(new_tuple)] = freq
    return new_splits

merge_rules = []
NUM_MERGES = 2

for merge_num in range(1, NUM_MERGES + 1):
    print(f"\n{'=' * 60}")
    print(f" MERGE {merge_num}")
    print(f"{'=' * 60}")
    
    # Count all pairs
    pair_counts = count_pairs(splits)
    sorted_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])
    
    print("\nAll adjacent pair frequencies:")
    for pair, count in sorted_pairs:
        marker = " ◀ HIGHEST" if count == sorted_pairs[0][1] and pair == sorted_pairs[0][0] else ""
        print(f"  ({pair[0]}, {pair[1]}): {count}{marker}")
    
    # Pick most frequent pair
    best_pair = sorted_pairs[0][0]
    best_count = sorted_pairs[0][1]
    merged_token = best_pair[0] + best_pair[1]
    
    print(f"\n>>> Merge Rule {merge_num}: '{best_pair[0]}' + '{best_pair[1]}' → '{merged_token}' (frequency: {best_count})")
    merge_rules.append((best_pair[0], best_pair[1], merged_token))
    
    # Apply merge
    splits = merge_pair(splits, best_pair)
    
    print(f"\nCorpus after Merge {merge_num}:")
    for token_tuple, freq in sorted(splits.items(), key=lambda x: -x[1]):
        print(f"  {' '.join(token_tuple):30s} (freq: {freq})")

In [ ]:
# ─────────────────────────────────────────────
# Final Summary
# ─────────────────────────────────────────────
print("=" * 60)
print(" FINAL RESULTS AFTER 2 BPE MERGES")
print("=" * 60)

print("\nMerge Rules:")
for i, (a, b, merged) in enumerate(merge_rules, 1):
    print(f"  Rule {i}: '{a}' + '{b}' → '{merged}'")

# Final vocabulary
final_vocab = set()
for token_tuple in splits:
    for tok in token_tuple:
        final_vocab.add(tok)

print(f"\nFinal Vocabulary ({len(final_vocab)} tokens):")
print(f"  {sorted(final_vocab)}")

print(f"\nFinal Tokenized Corpus:")
for token_tuple, freq in sorted(splits.items(), key=lambda x: -x[1]):
    print(f"  {' '.join(token_tuple):30s} (freq: {freq})")

### How BPE Handles Out-of-Vocabulary (OOV) Words

BPE **eliminates the OOV problem** entirely. Here is how:

1. **Base vocabulary includes all individual characters.** Since any word is composed of characters, BPE can always fall back to character-level tokenization for any unseen word.

2. **Learned merge rules are applied greedily.** When encountering a new, unseen word at inference time, BPE:
   - Splits the word into individual characters.
   - Applies the learned merge rules iteratively (in the order they were learned) to combine frequent subword pairs.
   - The result is a sequence of subword units, some of which may be single characters.

3. **Example:** If the unseen word `"lower"` appears and we have learned `lo` and `low` as subwords:
   - Start: `l o w e r </w>`
   - After Rule 1 (`l+o→lo`): `lo w e r </w>`
   - After Rule 2 (`lo+w→low`): `low e r </w>`
   - Final tokens: `['low', 'e', 'r', '</w>']`

> **Key Advantage:** Unlike word-level vocabularies that map unseen words to a single `<UNK>` token (losing all information), BPE preserves partial morphological information through shared subword units, enabling the model to generalize to novel words.